# SAÉ — Migration vers NoSQL

## Contexte et contraintes métier

La base décrit la programmation d’une radio et ses diffusions. Les requêtes s’appuient notamment sur les règles suivantes :

- **Protection du jeune public** : aucun titre `explicit` ne doit être diffusé pendant les tranches protégées du matin et de l’après-midi.
- **Quota de nouveautés** : l’antenne doit diffuser un minimum de titres récent·e·s, sélectionné·e·s selon la date de sortie de leur album.
- **Déclaration des diffusions** : pour les sociétés de gestion de droits, il faut pouvoir obtenir rapidement, pour chaque artiste, le nombre de diffusions et la durée cumulée correspondante.

Pour chaque requête, choisissez un ordre de présentation lorsqu’il n’est pas évident, afin de pouvoir comparer les résultats après migration.

In [7]:
import sqlite3
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Connection to Radio Kiosque database
sqlite_connection = sqlite3.connect("data/RadioKiosque.sqlite")

In [10]:
!pip install pandas --quiet


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Comment requêter la base de données ?

In [11]:
pd.read_sql_query(
    """
        SELECT host, title
        FROM Shows
        ORDER BY host
    """,
    con=sqlite_connection
)

,host,title
0,Anouk Lindqvist,Le Réveil Kiosque
1,Anouk Lindqvist,Rush Hour
2,DJ Marvelle,Les Insomniaques
3,Dario Fenn,Tour de Piste
4,Françoise Dubreuil,Dimanche Vinyle
5,Karim Benali,Café Croissant
6,Karim Benali,Nouveautés Kiosque
7,Orla Quinn,Pause Midi
8,Orla Quinn,Weekend Matin
9,Salomé Ndiaye,Scène Ouverte


### 1. Auditeur·ices sans demande

Lister les auditeur·ices n’ayant jamais effectué de demande musicale.

In [18]:
pd.read_sql_query(
    """
        SELECT listener_id, pseudo, country, city
        FROM Listeners
        WHERE listener_id NOT IN (
            SELECT listener_id 
            FROM Requests)
        ;
    """,
    con=sqlite_connection
)


,listener_id,pseudo,country,city
0,3,solar_raven63,Canada,Québec
1,7,red_seal79,France,Nantes
2,10,lunar_llama20,Belgique,Liège
3,12,pink_lynx87,Belgique,Bruxelles
4,14,red_owl90,Allemagne,Berlin
...,...,...,...,...
174,593,big_owl10,France,Lille
175,594,secret_sparrow56,Italie,Milan
176,596,fast_sparrow44,France,Grenoble
177,598,calm_raven24,Royaume-Uni,Londres


### 2. Bilan par animateur·ice

Pour chaque animateur·ice, indiquer le nombre d’émissions, le nombre de diffusions et la durée totale diffusée.

In [20]:
pd.read_sql_query(
    """
        SELECT *
        FROM Show
        ;
    """,
    con=sqlite_connection
)

,show_id,title,host,time_slot,is_protected
0,1,Le Réveil Kiosque,Anouk Lindqvist,morning,1
1,2,Café Croissant,Karim Benali,morning,1
2,3,Pause Midi,Orla Quinn,afternoon,1
3,4,Tour de Piste,Dario Fenn,afternoon,1
4,5,Rush Hour,Anouk Lindqvist,evening,0
5,6,Nouveautés Kiosque,Karim Benali,evening,0
6,7,Scène Ouverte,Salomé Ndiaye,evening,0
7,8,Rap Game,Yasmine Tazi,evening,0
8,9,Nuit Blanche,Salomé Ndiaye,night,0
9,10,Les Insomniaques,DJ Marvelle,night,0


In [19]:
pd.read_sql_query(
    """
        SELECT host, pseudo, country, city
        FROM Listeners
        WHERE host NOT IN (
            SELECT listener_id 
            FROM Requests)
        ;
    """,
    con=sqlite_connection
)

DatabaseError: Execution failed on sql '
        SELECT host, pseudo, country, city
        FROM Listeners
        WHERE host NOT IN (
            SELECT listener_id 
            FROM Requests)
        ;
    ': no such column: host

### 3. Bilan par tranche horaire

Pour chaque tranche horaire, indiquer le nombre de diffusions et le nombre de titres explicites diffusés.

### 4. Bilan par titre

Pour chaque titre, indiquer le nombre de diffusions, l’audience cumulée et le nombre d’émissions différentes où il a été programmé.

### 5. Bilan par artiste

Pour chaque artiste (principal·e ou invité·e), indiquer le nombre de titres au catalogue, le nombre de diffusions et la durée cumulée diffusée. Inclure les artistes jamais diffusé·es. Trier par ordre décroissant du nombre de diffusions.

### 6. Diffusions par décennie et tranche horaire

Présenter le nombre de diffusions sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

Voici le format attendu :

| decade | morning | afternoon | evening | night |
|-------:|--------:|----------:|--------:|------:|
|   1960 |         |           |         |       |
|   1970 |         |           |         |       |
|   1980 |         |           |         |       |
|   1990 |         |           |         |       |
|   2000 |         |           |         |       |
|   2010 |         |           |         |       |
|   2020 |         |           |         |       |

&#128161; Exécuter la requête en SQL avec la décennie, la tranche horaire et le nombre de diffusion. Puis pivoter la table avec la fonction `pivot_table` de Pandas.

### 7. Audience par décennie et tranche horaire

Présenter l’audience cumulée sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

### 8. Top 10 des audiences moyennes

Parmi les titres diffusés au moins trois fois, lister les dix titres ayant la plus forte audience moyenne par diffusion. Trier par l'audience moyenne.

### 9. Infractions à la protection du jeune public

Lister chaque diffusion d’un titre explicite dans une émission protégée, avec le titre, l’identifiant de l’émission et l’horodatage.

### 10. Albums diffusés en entier

Lister les albums dont tous les titres ont été diffusés au moins une fois.

### 11. Podcasts classés par pays

Pour chaque pays, indiquer le nombre de podcasts classés, le nombre de podcasts explicites et le nombre de podcasts dont la fiche est fiable.

### 12. Podcasts programmables

Lister les podcasts programmables, avec leur nom, leur éditeur, le nombre de pays dans le top 10 desquels ils figurent et leur meilleur rang. Trier du plus présent au moins présent, puis par meilleur rang.

### 13. Durée et activité des podcasts programmables

Pour chaque podcast programmable, indiquer le nombre d’épisodes connus, la durée moyenne de ses épisodes (en minutes) et la date de son dernier épisode. Ne garder que les podcasts dont la durée moyenne ne dépasse pas une heure.

In [ ]:
# Close the connection to the database
sqlite_connection.close()